# <font color="orange">Aiko Chat Tutorial: Robots have joined the Chat</font>

This tutorial shows how to create a simple chat agent for a robot in Aiko Chat.

<font color="orange"><i>2026-06-08: version 0</i></font>

## <font color="orange">Running this against the aiko_turtle sandbox (2026-11)</font>

This is the June 2026 tutorial notebook, unchanged except for the broker name and these notes. It drives a robot that speaks the tutorial's `turtle_robot:0` interface. The sandbox at https://turtle.aiko.hashtag.ai creates virtual turtles that speak exactly that interface, so this notebook works without a part-2 notebook running:

1. On the sandbox page, create a turtle. Its default interface is `turtle_robot:0 (legacy)`. Give it a name nobody else will pick, e.g. `my_turtle_yourname`.
2. Below, set `_ROBOT_NAME = "my_turtle_yourname"` (the cell that creates the chatbot). Leave `_HOST = "*"`.
3. Run every cell. `chatbot.robot_server.forward(100)` moves the turtle on the sandbox map, and so does a chat message `@@colab_virtualbot_YOU forward 100` in the `#robots` channel (the sandbox page shows that channel; the ChatServer is the shared one on `aiko.hashtag.ai`).

Run it in Google Colab: File > Upload notebook (or open it from GitHub: `anjsimmo/aiko_turtle`, path `examples/notebooks`). Runtime > Run all. The first two install cells take about a minute. If the broker name in your copy still says `mqtt.hashtag.ai`, change it to `aiko.hashtag.ai`.


## <font color="orange">Install the Aiko Services Python package</font>

Colab limitation: If running Aiko Services on your own computer, it's usually best to install it in developer (edit) mode with `pip install -e .` so that you can easily make changes. Though due to a limitation of Colab, the edit mode doesn't seem to work well together with Aiko Chat, so we need to `pip install .` instead. We won't be making any changes to any Aiko in this notebook, but if you do, you'll need to completely uninstall and reinstall the packages you modify.

In [ ]:
%cd /content
!git clone https://github.com/geekscape/aiko_services.git
%cd /content/aiko_services
!echo
!du -sh .
%ls

In [ ]:
# This cell takes around 20 seconds to run

!sed -i '/Pillow/d' pyproject.toml  # Avoid session restart due to older package version
!sed -i '/psutil/d' pyproject.toml  # Ditto
%pip install .

## <font color="orange">Install the Aiko Chat Python package</font>

In [ ]:
%cd /content
!git clone https://github.com/geekscape/aiko_chat.git
%cd /content/aiko_chat
!echo
!du -sh .
%ls

In [ ]:
# (aiko_chat no longer imports aiko_services.examples; this sed is kept in case an
# older checkout does)
# The Aiko Services pyproject.toml build file excludes the aiko_services.examples package.
# Since we don't need to actually run the Chat Server inside of Colab, we can comment references to
# aiko_services.examples package out to avoid the error:
# No module named 'aiko_services.examples'
!sed -i 's/from aiko_services.examples/#from aiko_services.examples/' src/aiko_chat/chat.py
%pip install .

In [ ]:
# Sanity check that we can import the packages we installed without errors or missing modules
import aiko_services as aiko
import aiko_chat
result = "succeeded" if "ChatServer" in dir(aiko_chat) else "failed"
print(f"Aiko Services and Aiko Chat installation: {result}")

## <font color="orange">Create your own custom Aiko Chatbot that runs on a background thread</font>

We can extend/compose bot.ChatBot to do some custom behaviour. In this case, we'll put any messages we receive into a queue and decide what to do with them later.


In [ ]:
import socket
DEFAULT_USERNAME = f"{socket.gethostname()[:7]}" # default username, by default will be colab notebook id

In [ ]:
### CHANGE YOUR USER NAME HERE, DO NOT USE QUOTES ###
!echo Change USER or it will appear as $DEFAULT_USERNAME
%env USER=$DEFAULT_USERNAME

%env AIKO_LOG_LEVEL=DEBUG
%env AIKO_MQTT_HOST=aiko.hashtag.ai
%cd /content/aiko_services/src/aiko_services/examples/colab

In [ ]:
import aiko_services as aiko
from abc import abstractmethod

class TurtleRobot(aiko.Actor):
    @abstractmethod
    def action(self, value):
        pass

    @abstractmethod
    def forward(self, value):
        pass

    @abstractmethod
    def left(self, value):
        pass

    @abstractmethod
    def right(self, value):
        pass

    @abstractmethod
    def backward(self, value):
        pass

In [ ]:
# This cell creates the chatbot running on a background thread

import os
from queue import Queue
from threading import Thread
import aiko_chat.bot as bot

botname = f"@@colab_virtualbot_{os.environ["USER"]}"  # CHANGE THIS!!!!
tags = ["ec=true"] # you can set any tags you like to make it easy to search for in the registrar

# Aiko services uses the composition pattern to create a service
_VERSION = 0
_ACTOR_BOT = "turtle_agent"
_PROTOCOL_BOT = f"{aiko.SERVICE_PROTOCOL_AIKO}/{_ACTOR_BOT}:{_VERSION}"
init_args = aiko.actor_args(_ACTOR_BOT, protocol=_PROTOCOL_BOT, tags=tags)
init_args["botname"] = botname

import queue
q = queue.Queue()

_ROBOT_NAME = "turtle_robot" # IMPORTANT: Rename your virtual robot to something unique so you can find it
_HOST = "*" # IMPORTANT: If your robot name isn't unique, then filter by host (of the robot)
assert _HOST != "*" or _ROBOT_NAME != "turtle_robot" # IMPORTANT: Either change the robot name or filter by host!

class ColabChatBot(bot.ChatBot):
    def __init__(self, context: bot.ChatBot, botname: str):
        context.call_init(self, "ChatBot", context)
        self.botname = botname
        self.robot_server = None

        print(f"Searching for {_ROBOT_NAME}")
        service_discovery, service_discovery_handler = aiko.do_discovery(
            TurtleRobot, aiko.ServiceFilter("*", _ROBOT_NAME, _HOST, "*", "*", "*"),
            self.discovery_add_robot_handler,
            self.discovery_remove_robot_handler)

    def process_message(self, payload_in, **kwargs):
        q.put(payload_in)

    def discovery_add_robot_handler(self, service_details, service):
        print(f"Connected    {service_details[1]}: {service_details[0]}")
        self.robot_server = service
        self.robot_server_topic = f"{service_details[0]}/in"

    def discovery_remove_robot_handler(self, service_details):
        print(f"Disconnected {service_details[1]}: {service_details[0]}")
        self.robot_server = None

chatbot = aiko.compose_instance(ColabChatBot, init_args)

thread = Thread(target=chatbot.run, args=(False,))
thread.daemon = True
thread.start()

In [ ]:
print(f"Botname is: {chatbot.botname}")

In [ ]:
import time; time.sleep(2.0) # give some time to discover the robot

In [ ]:
print(chatbot.chat_server)

In [ ]:
chatbot.robot_server

In [ ]:
# You can directly run actions like this
chatbot.robot_server.action("forward")
chatbot.robot_server.backward(100)

In [ ]:
import json

# This cell will process messages until stopped
# chatbot.change_channel("robot") # TODO: Fix bug in change_channel
print(f"In channel: {chatbot.current_channel}")

while True:
    payload = q.get()
    print(f"Received: {payload}")

    message = json.loads(payload)["message"]
    if chatbot.botname in message:
         print(f"Mentioned in: {payload}")
         recipients = [chatbot.current_channel]
         chatbot.chat_server.send_message(
             chatbot.botname, recipients, f"Hello from {chatbot.botname[2:]}, command acknowledged!"
         )
         command = message.split(chatbot.botname)[-1].strip()
         parts = command.split(" ")

         match parts:
            case ("action", x):
                chatbot.robot_server.action(x)
            case ("forward", x):
                chatbot.robot_server.forward(x)
            case ("left", x):
                chatbot.robot_server.left(x)
            case ("right", x):
                chatbot.robot_server.right(x)
            case ("backward", x):
                chatbot.robot_server.backward(x)
            case _:
                # assume it's a direct action
                chatbot.robot_server.action(" ".join(parts))

    q.task_done()